# Block 1 — Document normalization (live `src/med_doc`)

Warp, section layout, crop-window gate. **Does not classify ticks.**

Open in Colab from branch **`block1`**. Do **not** upload clinic PHI — this notebook uses the committed blank form.


## 0. Clone the live tree


In [ ]:
"""Colab/kernel bootstrap for live src/med_doc.

Opening a GitHub notebook does not clone the repo. Run this as the first cell.
Prints BOOTSTRAP_V3 when import med_doc succeeds.
"""

from __future__ import annotations

import os
import site
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_VERSION = "BOOTSTRAP_V3"
REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"


def _run(cmd: list[str]) -> None:
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def _token() -> str | None:
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def _clone_url() -> str:
    tok = _token()
    if tok:
        return f"https://{tok}@github.com/RwaRwa599/epq3.git"
    return REPO


def _find_root() -> Path | None:
    here = Path.cwd().resolve()
    for cand in (
        here,
        here.parent,
        Path("/content/epq3"),
        Path("/content") / "epq3",
    ):
        if (cand / "src" / "med_doc" / "__init__.py").is_file():
            return cand
    return None


def _write_pth(src: Path) -> None:
    line = str(src.resolve()) + "\n"
    dirs = []
    try:
        dirs.extend(site.getsitepackages())
    except Exception:
        pass
    try:
        dirs.append(site.getusersitepackages())
    except Exception:
        pass
    sp = Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    dirs.append(str(sp))
    for d in dirs:
        if not d:
            continue
        target = Path(d)
        try:
            target.mkdir(parents=True, exist_ok=True)
            (target / "epq3_src.pth").write_text(line, encoding="utf-8")
            print("wrote", target / "epq3_src.pth")
        except Exception as exc:
            print("pth skip", target, exc)


def _ipython_cd(path: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is None:
        os.chdir(path)
        return
    ip.run_line_magic("cd", str(path))


def _ipython_pip(root: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is not None:
        ip.run_line_magic("pip", "install -q matplotlib opencv-python-headless pydantic Pillow numpy")
        ip.run_line_magic("pip", f"install -q -e {root}")
        return
    _run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "opencv-python-headless", "pydantic", "Pillow", "numpy"])
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)])


def put_src_on_path(root: Path | None = None) -> Path:
    root = root or _find_root()
    if root is None:
        raise ModuleNotFoundError(
            "med_doc not found. Run the first notebook cell (BOOTSTRAP_V3 clone). "
            "Private repo: Colab secret GITHUB_TOKEN. Then Runtime → Run all."
        )
    src = (root / "src").resolve()
    os.chdir(root)
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
    os.environ["PYTHONPATH"] = str(src) + os.pathsep + os.environ.get("PYTHONPATH", "")
    return root


def bootstrap() -> Path:
    print(BOOTSTRAP_VERSION)
    dest = Path("/content/epq3") if Path("/content").is_dir() else (Path.cwd().resolve() / "epq3")
    root = _find_root()
    if root is None:
        url = _clone_url()
        if dest.exists() and not (dest / "src" / "med_doc" / "__init__.py").is_file():
            import shutil

            shutil.rmtree(dest, ignore_errors=True)
        if not (dest / ".git").is_dir():
            _run(["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", url, str(dest)])
        else:
            _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
            _run(["git", "-C", str(dest), "checkout", BRANCH])
            _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
        root = dest
    _ipython_cd(root)
    os.chdir(root)
    src = root / "src"
    if str(src.resolve()) not in sys.path:
        sys.path.insert(0, str(src.resolve()))
    _write_pth(src)
    try:
        _ipython_pip(root)
    except Exception as exc:
        print("pip note:", exc)
    # Drop a copy next to cwd as last resort (some Colab kernels ignore .pth until restart)
    try:
        import med_doc  # noqa: F401
    except ModuleNotFoundError:
        sys.path.insert(0, str(src.resolve()))
        import importlib

        importlib.invalidate_caches()
        import med_doc  # noqa: F401
    import med_doc

    print("cwd:", os.getcwd())
    print("med_doc:", med_doc.__file__)
    if "src" not in Path(med_doc.__file__).parts:
        print("warning: unexpected med_doc location")
    return root


root = bootstrap()


## 1. Helpers


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

## 1b. Block 1a batch (folder, ZIP, or many uploads)

`run_block1a_batch` warps many photos **without** 1b/1c crops. For the **full** Blocks 1–5 batch (normalize → drafts → KG → LIS), use `run_blocks_1_to_5` in `Pipeline_Blocks_1_to_5.ipynb` with the same folder / ZIP / upload input.

Do **not** upload clinic PHI. Default below uses two copies of the synthetic blank.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from med_doc.normalization.block1a import run_block1a_batch

# Colab: set True and pick multiple images (or one .zip).
USE_UPLOAD = False
batch_input = None
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        batch_input = names
else:
    sheet = demo_sheet()
    batch_dir = OUT / "raw_1a"
    batch_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(sheet, batch_dir / "blank_a.png")
    shutil.copy2(sheet, batch_dir / "blank_b.png")
    batch_input = batch_dir

b1a = run_block1a_batch(
    batch_input,
    output_dir=OUT / "b1a",
    output_zip=OUT / "block1a.zip",
)
print("1a docs", b1a["manifest"]["successful_documents"], b1a["output_zip"])
for row in b1a["manifest"]["documents"]:
    print(" ", row.get("doc_id"), row.get("status"), row.get("warp_method"), row.get("canvas_size"))

ok = [d for d in b1a["manifest"]["documents"] if d.get("status") == "success"]
if ok:
    show_rgb(OUT / "b1a" / "docs" / ok[0]["doc_id"] / "canonical.png", f"1a {ok[0]['doc_id']}")
download(OUT / "block1a.zip")


## 2. Normalize the synthetic blank

`normalize_document` returns a canonical canvas + checkbox/handwriting crops. `normalize_batch` writes `block1_normalized_batch.zip` for Blocks 3–5.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from med_doc.normalization import normalize_document
from med_doc.normalization.batch import normalize_batch

sheet = demo_sheet()
print("input:", sheet)

result = normalize_document(str(sheet), document_id=sheet.stem)
print("template:", result.extra.get("template_id"))
print("warp:", result.warp_method, "orientation:", result.orientation_degrees)
print("alignment:", round(float(result.alignment_confidence), 3))
print("checkboxes:", len(result.checkbox_crops), "handwriting:", len(result.handwriting_crops))

b1 = normalize_batch([sheet], output_dir=OUT / "b1", output_zip=OUT / "block1.zip")
print("ZIP:", b1["output_zip"], "ok", b1["manifest"]["successful_documents"])

## 3. Canonical canvas and overlay


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(result.canonical_canvas)
axes[0].set_title(f"canonical {result.canonical_canvas.shape[1]}x{result.canonical_canvas.shape[0]}")
axes[0].axis("off")
if result.debug_overlay is not None:
    axes[1].imshow(result.debug_overlay)
    axes[1].set_title("debug overlay")
else:
    axes[1].axis("off")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Crop gallery


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

cb_items = list(result.checkbox_crops.items())[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, (fid, crop) in zip(axes.ravel(), cb_items):
    ax.imshow(crop.normalized_image)
    ax.set_title(f"{fid}\nq={crop.quality_score:.2f}", fontsize=8)
    ax.axis("off")
plt.suptitle("Checkbox crops (Block 1 does not label ticks)")
plt.tight_layout()
plt.show()

hw_items = list(result.handwriting_crops.items())[:6]
fig, axes = plt.subplots(len(hw_items), 1, figsize=(10, 2.2 * max(len(hw_items), 1)))
if len(hw_items) == 1:
    axes = [axes]
for ax, (fid, crop) in zip(axes, hw_items):
    ax.imshow(crop.normalized_image)
    ax.set_title(fid)
    ax.axis("off")
plt.suptitle("Handwriting ROIs")
plt.tight_layout()
plt.show()

## 5. Download Block 1 ZIP


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

download(OUT / 'block1.zip')